# 04 Experiment Sandbox: Hyperparameter Workbench

Use this notebook to test variants before committing settings to YAML. The intended workflow is:

1. Run the official pipeline only up to the upstream artifact needed for an experiment.
2. Run the relevant experiment section here.
3. Inspect the compact `*_summary.csv` or `*_summary.md` in `outputs/<mode>/experiments/<experiment_name>/`.
4. Promote only the winning hyperparameters into `configs/dev.yaml` or `configs/base.yaml`.
5. Rerun `03_ml_pipeline.ipynb` as the clean official pipeline.

Sandbox outputs are evidence, not final deliverables.


## Setup

Set `RUN_MODE`, load the prepared transaction scan, and point the notebook at existing official pipeline artifacts. Missing artifacts are expected if you have not run the required upstream stage yet.


In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import Markdown, display

# Mode toggle: experiments should usually run in dev first.
RUN_MODE = "dev"  # change to "prod" only when intentionally experimenting on full data
os.environ["CARREFOUR_MODE"] = RUN_MODE.strip().lower()

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode
from src.data_loader import load_prepared_transactions
from src.utils import set_global_seed

CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
OUTPUTS = CONFIG.outputs

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()
transactions = load_prepared_transactions(cfg=CONFIG)

basket_path = CONFIG.artifact_path("baskets", "output", directory=CONFIG.outputs / "embeddings")
product_embeddings_path = CONFIG.artifact_path("word2vec", "embeddings_output", directory=CONFIG.outputs / "embeddings")
customer_embeddings_path = CONFIG.artifact_path("customer_embeddings", "output", directory=CONFIG.outputs / "features")
behavior_path = CONFIG.artifact_path("behavioral_features", "output", directory=CONFIG.outputs / "features")
feature_sets = {
    name: CONFIG.outputs / "features" / filename
    for name, filename in CONFIG.get("feature_sets.outputs", {}).items()
}

artifact_status = {
    "basket_path": basket_path,
    "product_embeddings_path": product_embeddings_path,
    "customer_embeddings_path": customer_embeddings_path,
    "behavior_path": behavior_path,
    **{f"feature_set:{name}": path for name, path in feature_sets.items()},
}

print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Experiment output path: {CONFIG.experiments}")
print(f"Experiments enabled: {CONFIG.experiments_enabled}")
if not CONFIG.experiments_enabled:
    display(Markdown("**Experiments are disabled for this mode.** Switch `RUN_MODE` to `dev` for sandbox runs."))
for label, path in artifact_status.items():
    status = "exists" if path.exists() else "missing"
    print(f"{label}: {status} | {path}")


## Experiment 2: Item2Vec Training

Optional sandbox for training Item2Vec variants without changing YAML. Edit the trial list in the cell below. This stage only trains models and writes product embedding tables; Experiment 3 evaluates those tables.

In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import run_item2vec_training_experiments

RUN_ITEM2VEC_TRAINING_SANDBOX = False
item2vec_sandbox_experiment_name = "item2vec_embedding_sandbox"
item2vec_sandbox_trials = [
    {
        "name": "current_config",
        "word2vec": {},
    },
    {
        "name": "dims100_epochs5",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
        },
    },
    {
        "name": "dims128_epochs5",
        "word2vec": {
            "vector_size": 128,
            "epochs": 5,
        },
    },
    {
        "name": "dims100_min_count5",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "min_count": 5,
        },
    },
    {
        "name": "dims100_subsample0005",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "sample": 0.0005,
        },
    },
    {
        "name": "dims100_min_count5_subsample0005",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "min_count": 5,
            "sample": 0.0005,
        },
    },
    {
        "name": "dims100_negative15",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "negative": 15,
        },
    },
]

if RUN_ITEM2VEC_TRAINING_SANDBOX:
    item2vec_training_sandbox = run_item2vec_training_experiments(
        basket_path,
        trials=item2vec_sandbox_trials,
        experiment_name=item2vec_sandbox_experiment_name,
        force=False,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Item2Vec Training Experiment

- Manifest parquet: `{item2vec_training_sandbox['manifest']['parquet']}`
- Manifest summary CSV: `{item2vec_training_sandbox['manifest']['summary_csv']}`
- Manifest summary Markdown: `{item2vec_training_sandbox['manifest']['summary_md']}`
"""))
    display(pl.read_parquet(item2vec_training_sandbox["manifest"]["parquet"]))
else:
    display(Markdown(
        "Item2Vec training sandbox is self-contained and currently disabled. "
        "Set `RUN_ITEM2VEC_TRAINING_SANDBOX = True` in this cell to train variants."
    ))


## Experiment 3: Product Embedding Validation

Optional sandbox for evaluating product embedding tables. It can evaluate the current pipeline embedding table, freshly trained Experiment 2 outputs, or existing experiment files already saved under `outputs/<mode>/experiments/<experiment_name>`.


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import evaluate_product_embedding_experiments

RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = False
embedding_validation_sandbox_experiment_name = globals().get(
    "item2vec_sandbox_experiment_name",
    "item2vec_embedding_sandbox",
)
embedding_validation_sandbox_sample_size = 300 if CONFIG.mode == "dev" else 500
embedding_validation_sandbox_neighbors = 8

if "item2vec_training_sandbox" in globals():
    embedding_validation_sandbox_paths = item2vec_training_sandbox["embedding_paths"]
else:
    embedding_validation_sandbox_paths = {"current_pipeline": product_embeddings_path}
    experiment_dir = CONFIG.experiments / embedding_validation_sandbox_experiment_name
    for trial in globals().get("item2vec_sandbox_trials", []):
        trial_name = trial["name"]
        candidate_path = experiment_dir / f"{trial_name}_product_embeddings.parquet"
        if candidate_path.exists():
            embedding_validation_sandbox_paths[trial_name] = candidate_path

if RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX:
    embedding_validation_sandbox = evaluate_product_embedding_experiments(
        embedding_validation_sandbox_paths,
        experiment_name=embedding_validation_sandbox_experiment_name,
        sample_size=embedding_validation_sandbox_sample_size,
        neighbors=embedding_validation_sandbox_neighbors,
        transactions=transactions,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Product Embedding Validation Experiment

- Diagnostics parquet: `{embedding_validation_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{embedding_validation_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{embedding_validation_sandbox['diagnostics']['summary_md']}`
- Best trial: `{embedding_validation_sandbox['best']['trial_name']}`
"""))
    display(
        pl.read_parquet(embedding_validation_sandbox["diagnostics"]["parquet"]).select([
            "embedding_sandbox_rank",
            "trial_name",
            "vector_dims",
            "product_vocab_coverage_pct",
            "same_sector_at_1_pct",
            "same_sector_neighbor_share_pct",
            "mean_neighbor_cosine",
            "embedding_quality_score",
            "selected_in_embedding_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Product embedding validation sandbox is self-contained and currently disabled. "
        "Set `RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = True` in this cell to compare embedding tables."
    ))


## Experiment 4: Customer Embedding Aggregation

Optional sandbox for testing how product embeddings are aggregated into customer vectors. Official clustering uses product quantities only. IDF variants are tested here before any promotion to YAML.


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.data_loader import load_prepared_transactions
from src.experiment_sandbox import run_customer_embedding_experiments

RUN_CUSTOMER_EMBEDDING_SANDBOX = False
customer_embedding_sandbox_experiment_name = "customer_embedding_sandbox"
customer_embedding_sandbox_sample_size = 5000 if CONFIG.mode == "dev" else 12000
customer_embedding_sandbox_trials = [
    {"name": "quantity_weighted", "weight_strategy": "quantity", "normalize_vectors": False},
    {"name": "quantity_idf_weighted", "weight_strategy": "quantity_idf", "normalize_vectors": False},
    {"name": "equal_weighted", "weight_strategy": "equal", "normalize_vectors": False},
    {"name": "equal_idf_weighted", "weight_strategy": "equal_idf", "normalize_vectors": False},
    {"name": "quantity_weighted_l2", "weight_strategy": "quantity", "normalize_vectors": True},
    {"name": "quantity_idf_weighted_l2", "weight_strategy": "quantity_idf", "normalize_vectors": True},
    {"name": "equal_weighted_l2", "weight_strategy": "equal", "normalize_vectors": True},
]

if "product_embeddings_path" not in globals():
    product_embeddings_path = CONFIG.artifact_path(
        "word2vec",
        "embeddings_output",
        directory=CONFIG.outputs / "embeddings",
    )
    if not product_embeddings_path.exists():
        raise FileNotFoundError(
            "Experiment 4 needs product embeddings. Run Stage 2 first or point "
            f"`product_embeddings_path` to an existing file. Missing: {product_embeddings_path}"
        )

if "transactions" not in globals():
    transactions = load_prepared_transactions(cfg=CONFIG)

if RUN_CUSTOMER_EMBEDDING_SANDBOX:
    customer_embedding_sandbox = run_customer_embedding_experiments(
        product_embeddings_path,
        trials=customer_embedding_sandbox_trials,
        experiment_name=customer_embedding_sandbox_experiment_name,
        sample_size=customer_embedding_sandbox_sample_size,
        transactions=transactions,
        force=True,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Customer Embedding Experiment Results

- Diagnostics parquet: `{customer_embedding_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{customer_embedding_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{customer_embedding_sandbox['diagnostics']['summary_md']}`
- Best trial: `{customer_embedding_sandbox['best']['trial_name']}`
"""))
    display(
        pl.read_parquet(customer_embedding_sandbox["diagnostics"]["parquet"]).select([
            "customer_embedding_sandbox_rank",
            "trial_name",
            "weight_strategy",
            "normalize_vectors",
            "vector_dims",
            "finite_pct",
            "zero_vector_pct",
            "effective_dimension",
            "mean_nearest_neighbor_cosine",
            "customer_embedding_quality_score",
            "selected_in_customer_embedding_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Customer embedding sandbox is self-contained and currently disabled. "
        "Set `RUN_CUSTOMER_EMBEDDING_SANDBOX = True` in this cell to compare customer-vector aggregation variants."
    ))


## Experiment 5: Customer Vector / Feature-Set Probes

Optional sandbox for comparing customer-vector feature variants before the full candidate model suite. It uses quick KMeans probes near the client hypothesis range and reports separability, balance, dimensional health, and finite-value checks.

In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import run_feature_set_experiments
from src.feature_engineering import build_feature_set

RUN_FEATURE_SET_SANDBOX = False
feature_set_sandbox_experiment_name = "customer_feature_set_sandbox"
feature_set_sandbox_sample_size = 5000 if CONFIG.mode == "dev" else 12000
feature_set_sandbox_k_values = [10, 12, 15]

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Experiment 5 needs the official Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

feature_set_sandbox_inputs = dict(feature_sets)
if "customer_embedding_sandbox" in globals():
    stage4b_experiment_dir = CONFIG.experiments / customer_embedding_sandbox_experiment_name
    for trial_name, embedding_path in customer_embedding_sandbox["customer_embedding_paths"].items():
        embeddings_only_path = build_feature_set(
            embedding_path,
            variant="embeddings_only",
            output_path=stage4b_experiment_dir / f"{trial_name}_feature_set_embeddings_only.parquet",
            force=False,
            cfg=CONFIG,
        )
        feature_set_sandbox_inputs[f"{trial_name}_embeddings_only"] = embeddings_only_path

        if "behavior_path" in globals():
            behavior_path_for_trial = build_feature_set(
                embedding_path,
                behavior_path=behavior_path,
                variant="embeddings_behavior",
                output_path=stage4b_experiment_dir / f"{trial_name}_feature_set_embeddings_behavior.parquet",
                force=False,
                cfg=CONFIG,
            )
            feature_set_sandbox_inputs[f"{trial_name}_embeddings_behavior"] = behavior_path_for_trial

if RUN_FEATURE_SET_SANDBOX:
    feature_set_sandbox = run_feature_set_experiments(
        feature_set_sandbox_inputs,
        experiment_name=feature_set_sandbox_experiment_name,
        k_values=feature_set_sandbox_k_values,
        sample_size=feature_set_sandbox_sample_size,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Feature-Set Experiment Results

- Diagnostics parquet: `{feature_set_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{feature_set_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{feature_set_sandbox['diagnostics']['summary_md']}`
- Best feature set: `{feature_set_sandbox['best']['feature_set_name']}` at k=`{feature_set_sandbox['best']['k']}`
"""))
    display(
        pl.read_parquet(feature_set_sandbox["diagnostics"]["parquet"]).select([
            "feature_set_sandbox_rank",
            "feature_set_name",
            "k",
            "feature_count",
            "effective_dimension",
            "silhouette",
            "davies_bouldin",
            "cluster_size_cv",
            "finite_pct",
            "feature_set_quality_score",
            "selected_in_feature_set_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Feature-set sandbox is self-contained and currently disabled. Set `RUN_FEATURE_SET_SANDBOX = True` "
        "in this cell when you want to compare pipeline feature variants. If Experiment 4 ran in this kernel, "
        "its customer-vector trials are included automatically."
    ))


## Experiment 6: Focused UMAP-HDBSCAN Trials

Use this optional cell when you want to test UMAP-HDBSCAN settings without rerunning the full Stage 6 candidate suite. It reuses the existing Stage 5 feature set and writes all experiment artifacts directly under `outputs/<mode>/experiments/<experiment_name>`. Trial names describe density assumptions, not target cluster counts.


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.model_selection import run_umap_hdbscan_experiments
from src.visualization import plot_stage6_model_diagnostics

RUN_UMAP_HDBSCAN_SANDBOX = False

selection_feature_set = globals().get(
    "selection_feature_set",
    CONFIG.get("modeling.feature_set_for_selection", "embeddings_only"),
)

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Experiment 6 needs the official Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

manual_umap_hdbscan_trials = [
    {
        "name": "local_leaf",
        "umap": {
            "n_components": 20,
            "n_neighbors": 30,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 300,
            "min_samples": 1,
            "cluster_selection_method": "leaf",
        },
    },
    {
        "name": "balanced_eom",
        "umap": {
            "n_components": 20,
            "n_neighbors": 50,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 500,
            "min_samples": 1,
            "cluster_selection_method": "eom",
        },
    },
    {
        "name": "broad_eom",
        "umap": {
            "n_components": 15,
            "n_neighbors": 75,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 750,
            "min_samples": 1,
            "cluster_selection_method": "eom",
        },
    },
]

experiment_name = "umap_hdbscan_density_trials"

if RUN_UMAP_HDBSCAN_SANDBOX:
    experiment_suite = run_umap_hdbscan_experiments(
        feature_sets[selection_feature_set],
        trials=manual_umap_hdbscan_trials,
        experiment_name=experiment_name,
        force=True,
        cfg=CONFIG,
    )
    experiment_figure = plot_stage6_model_diagnostics(
        experiment_suite["diagnostics"]["parquet"],
        output_path=CONFIG.figures / f"stage6b_{experiment_name}_diagnostics.png",
        cfg=CONFIG,
    )

    display(Markdown(f"""
### Focused UMAP-HDBSCAN Experiment Results

- Diagnostics parquet: `{experiment_suite['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{experiment_suite['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{experiment_suite['diagnostics']['summary_md']}`
- Diagnostics figure: `{experiment_figure}`
"""))
    display(Image(filename=str(experiment_figure)))

    display(pl.read_parquet(experiment_suite["diagnostics"]["parquet"]).select([
        "stage6_rank",
        "model_name",
        "trial_name",
        "model_variant",
        "cluster_count",
        "coverage_adjusted_silhouette",
        "silhouette",
        "davies_bouldin",
        "noise_pct",
        "passes_quality_gate",
    ]))
else:
    display(Markdown(
        "Focused UMAP-HDBSCAN sandbox is self-contained and currently disabled. "
        "Set `RUN_UMAP_HDBSCAN_SANDBOX = True` in this cell to run these trials."
    ))
